# Evaluate A-OKVQA bằng `aokvqa_mc_eval.py` trên Kaggle

Notebook clone SelTDA, chuẩn bị annotation A-OKVQA từ Kaggle Input, dùng ảnh COCO 2017 read-only, rồi chạy `aokvqa_mc_eval.py` để chấm multiple-choice accuracy cho student checkpoint.

Cần Add Input: A-OKVQA dataset, COCO 2017 dataset, và dataset/output chứa `checkpoint_*.pth`. Output nằm trong `/kaggle/working/aokvqa_eval`.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/fantastichaha11/SelTDA.git'
BRANCH = 'feat/pseudo-label-filter'
REPO_DIR = Path('/kaggle/working/SelTDA')

AOKVQA_INPUT_ROOT = Path('/kaggle/input/datasets/phong2004/a-okvqa/aokvqa')
AOKVQA_ROOT = Path('/kaggle/working/data/aokvqa_eval')
COCO_INPUT_ROOT = Path('/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017')
OUTPUT_ROOT = Path('/kaggle/working/aokvqa_eval')
CONFIG_PATH = REPO_DIR / 'configs/aokvqa_eval_kaggle.yaml'
CONFIG_ARG = 'configs/aokvqa_eval_kaggle.yaml'

# Nếu muốn eval checkpoint cụ thể, điền list path ở đây. Nếu để rỗng, notebook tự tìm checkpoint_*.pth.
CHECKPOINT_PATHS = []
CHECKPOINT_SEARCH_ROOTS = [
    Path('/kaggle/working/student_vqascore'),
    Path('/kaggle/input'),
]
CHECKPOINT_GLOB = 'checkpoint_*.pth'
MAX_CHECKPOINTS = None  # Ví dụ 1 để chỉ eval checkpoint đầu tiên tìm được.

DEVICE = 'cuda'
BATCH_SIZE = 1
NUM_WORKERS = 2
SEED = 42

In [ ]:
if not REPO_DIR.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
    if _exit_code != 0:
        raise RuntimeError(f'git clone failed với exit code {_exit_code}')
else:
    print('Repo đã tồn tại, bỏ qua clone:', REPO_DIR)

!python -m pip install -q --upgrade omegaconf==2.3.0 hydra-core==1.3.2 timm==0.4.12 fairscale==0.4.13 transformers==4.36.1
if _exit_code != 0:
    raise RuntimeError(f'pip install failed với exit code {_exit_code}')

# Patch aokvqa_mc_eval.py: answer_ids là tensor, và mỗi sample có choices riêng nên eval từng sample trong batch.
eval_script = REPO_DIR / 'aokvqa_mc_eval.py'
text = eval_script.read_text()
old = '''    for n, (image, question, _question_id) in tqdm(
        enumerate(val_loader), total=len(val_loader), desc="A-OKVQA MC eval"
    ):
        ann = val_annotations[n]
        answer_list = ann["choices"]
        answer_candidates = prep_answer_candidates(model, answer_list, device=device)
        image = image.to(device, non_blocking=True)
        correct_answer = answer_list[ann["correct_choice_idx"]]

        answer_ids = model(
            image,
            question,
            answer_candidates,
            train=False,
            inference="rank",
            k_test=len(answer_list),
        )
        model_choices.append(answer_list[answer_ids])
        correct_choices.append(correct_answer)
'''
new = '''    sample_idx = 0
    progress = tqdm(total=len(val_loader.dataset), desc="A-OKVQA MC eval")
    for image, question, _question_id in val_loader:
        batch_size = int(image.shape[0])
        image = image.to(device, non_blocking=True)

        for b in range(batch_size):
            ann = val_annotations[sample_idx]
            answer_list = ann["choices"]
            answer_candidates = prep_answer_candidates(model, answer_list, device=device)
            correct_answer = answer_list[ann["correct_choice_idx"]]

            if isinstance(question, (list, tuple)):
                sample_question = [question[b]]
            else:
                sample_question = [question]

            answer_ids = model(
                image[b : b + 1],
                sample_question,
                answer_candidates,
                train=False,
                inference="rank",
                k_test=len(answer_list),
            )
            answer_id = int(answer_ids.detach().cpu().reshape(-1)[0].item())
            model_choices.append(answer_list[answer_id])
            correct_choices.append(correct_answer)
            sample_idx += 1
            progress.update(1)
    progress.close()
'''
if old in text:
    eval_script.write_text(text.replace(old, new))
    print('Patched:', eval_script)
elif 'answer_id = int(answer_ids.detach().cpu().reshape(-1)[0].item())' in text:
    print('aokvqa_mc_eval.py already patched')
else:
    raise RuntimeError('Không tìm thấy đoạn cần patch trong aokvqa_mc_eval.py')
print('Clone và dependency setup hoàn tất.')

In [ ]:
import json, shutil

if not AOKVQA_INPUT_ROOT.is_dir():
    raise FileNotFoundError(f'Không thấy AOKVQA_INPUT_ROOT: {AOKVQA_INPUT_ROOT}')
coco_val_root = COCO_INPUT_ROOT / 'val2017'
if not coco_val_root.is_dir():
    raise FileNotFoundError(f'Không thấy COCO val2017: {coco_val_root}')

AOKVQA_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(AOKVQA_INPUT_ROOT, AOKVQA_ROOT, dirs_exist_ok=True)

def load_json(path):
    with Path(path).open(encoding='utf-8') as f:
        return json.load(f)

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(data, f)
    tmp.replace(path)

def convert_aokvqa_record(record, split):
    image_file = f'{split}2017/{int(record["image_id"]):012d}.jpg'
    return {
        'dataset': 'aokvqa',
        'image': image_file,
        'question': record['question'],
        'question_id': record['question_id'],
        'answer': record.get('direct_answers'),
        'rationales': record.get('rationales'),
    }

for split in ('train', 'val'):
    raw_path = AOKVQA_ROOT / f'aokvqa_v1p0_{split}.json'
    raw_records = load_json(raw_path)
    converted = [convert_aokvqa_record(record, split) for record in raw_records]
    write_json(AOKVQA_ROOT / f'{split}.json', converted)
    print(f'Wrote {len(converted):,} records -> {AOKVQA_ROOT / f"{split}.json"}')

vocab_path = AOKVQA_ROOT / 'specialized_vocab_train.csv'
with vocab_path.open(encoding='utf-8') as f:
    answer_list = [line.strip() for line in f if line.strip()]
write_json(AOKVQA_ROOT / 'answer_list.json', answer_list)

val_records = load_json(AOKVQA_ROOT / 'val.json')
missing = [COCO_INPUT_ROOT / record['image'] for record in val_records if not (COCO_INPUT_ROOT / record['image']).is_file()]
if missing:
    raise FileNotFoundError(f'Thiếu {len(missing)} ảnh A-OKVQA val. Ví dụ: {missing[:5]}')
print(f'Đã xác minh {len(val_records):,} ảnh val trong {coco_val_root}')

In [ ]:
import yaml

config = {
    'vqa_root': str(COCO_INPUT_ROOT),
    'vg_root': str(COCO_INPUT_ROOT),
    'ann_root': str(AOKVQA_ROOT),
    'dataset_name': 'aokvqa',
    'train_files': ['train'],
    'truncate_train_dataset_to': None,
    'append_rationale_to_answer': False,
    'append_rationale_to_question': False,
    'use_rationale_as_answer': False,
    'use_validation_set_as_test_set': True,
    'pretrained': '',
    'vit': 'base',
    'batch_size_train': 1,
    'batch_size_test': BATCH_SIZE,
    'vit_grad_ckpt': False,
    'vit_ckpt_layer': 0,
    'init_lr': 2e-5,
    'image_size': 480,
    'k_test': 128,
    'inference': 'rank',
    'weight_decay': 0.05,
    'min_lr': 0,
    'max_epoch': 1,
    'torch_home': '/kaggle/working/torch_home',
    'wandb': False,
    'save_last_only': True,
}
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print(CONFIG_PATH.read_text())

In [ ]:
import re

def safe_label(path):
    path = Path(path)
    label = f'{path.parent.name}__{path.stem}'
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', label)

if CHECKPOINT_PATHS:
    candidates = [Path(path) for path in CHECKPOINT_PATHS]
else:
    candidates = []
    for root in CHECKPOINT_SEARCH_ROOTS:
        if root.exists():
            candidates.extend(root.rglob(CHECKPOINT_GLOB))

candidates = sorted({path.resolve() for path in candidates if path.is_file() and path.stat().st_size > 1024 * 1024}, key=lambda p: str(p))
if MAX_CHECKPOINTS is not None:
    candidates = candidates[:MAX_CHECKPOINTS]
if not candidates:
    raise FileNotFoundError('Không tìm thấy checkpoint_*.pth. Hãy set CHECKPOINT_PATHS hoặc attach Kaggle dataset chứa checkpoint.')

checkpoint_paths = {safe_label(path): path for path in candidates}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for label, path in checkpoint_paths.items():
    print(f'{label}: {path} ({path.stat().st_size / 1024**3:.2f} GiB)')

In [ ]:
%cd {REPO_DIR}
!python -c 'import aokvqa_mc_eval; print("aokvqa_mc_eval import OK")'
if _exit_code != 0:
    raise RuntimeError(f'aokvqa_mc_eval import failed với exit code {_exit_code}')

In [ ]:
%cd {REPO_DIR}
result_paths = []
for label, checkpoint in checkpoint_paths.items():
    out_path = OUTPUT_ROOT / label / 'aokvqa_mc_eval.json'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'=== Eval {label} ===')
    !python aokvqa_mc_eval.py --config={CONFIG_ARG} --checkpoint={checkpoint} --output={out_path} --device={DEVICE} --batch-size={BATCH_SIZE} --num-workers={NUM_WORKERS}
    if _exit_code != 0:
        raise RuntimeError(f'aokvqa_mc_eval failed cho {label} với exit code {_exit_code}')
    result_paths.append(out_path)
print('Done. Results:', result_paths)

In [ ]:
rows = []
for path in sorted(OUTPUT_ROOT.rglob('aokvqa_mc_eval.json')):
    with path.open(encoding='utf-8') as f:
        item = json.load(f)
    rows.append({
        'label': path.parent.name,
        'accuracy': item['accuracy'],
        'correct': item['correct'],
        'total': item['total'],
        'checkpoint': item['checkpoint'],
        'result_file': str(path),
    })
rows = sorted(rows, key=lambda row: row['accuracy'], reverse=True)
for row in rows:
    print(f"{row['label']}: {row['accuracy']:.4%} ({row['correct']}/{row['total']}) -> {row['result_file']}")